# Topic modeling (LDA) — exploratory pass

Explores what policy topics are discussed in state-parliament speeches, scoped to each state's
legislative period immediately before vs. after AfD entry. This does not measure morality or
politeness — the topic assigned to each speech is meant as a **control** variable for later
analysis (some topics may be more polarized/moralized than others, independent of AfD entry).

Runs two independent topic-count-selection methods and compares them: gensim LDA scored by c_v
coherence, and sklearn LDA scored by held-out log-likelihood (the latter per the practical guide
https://medium.com/data-science/practical-guide-to-topic-modeling-with-lda-05cd6b027bdf, which
argues coherence is unreliable for tuning -- rather than pick a side, both run and get compared).

See `docs/superpowers/specs/2026-08-14-topic-modeling-lda-design.md` (local-only, not tracked in
git) for full design rationale.

In [1]:
import os
import sys

import spacy
from dotenv import load_dotenv

sys.path.insert(0, os.path.abspath(".."))
load_dotenv("../.env")

from topic_modeling_lib import (
    build_gensim_corpus,
    build_sklearn_corpus,
    gensim_coherence_scan,
    load_cache,
    load_corpus,
    make_spacy_preprocessor,
    save_cache,
    sklearn_loglikelihood_search,
    sklearn_topic_words,
    timed,
)

DATA_ROOT = os.environ["DATA_ROOT"]
timing_log = []

## Parameters

Start small (single state, sampled) to get a fast timing read before scaling up -- see the
"Timing" cell at the end for what that read implies about a full run.

In [2]:
STATES = None       # None = all 16 states
PRE_POST = None        # "pre", "post", or None for both
SAMPLE_N = None         # None = use every speech in scope
SAMPLE_SEED = 42        # random_state for load_corpus's sampling
K_RANGE = [32, 34, 36, 38, 40, 42, 44, 46, 48]   # fine-grained around the k=40 gensim peak
SKLEARN_N_ITER = 9      # None = try every k in K_RANGE

# Cache key inputs for everything downstream. SAMPLE_SEED is in here because a different
# random sample of the same size is a different corpus, not a cache hit. "preprocessing" is
# filled in from the actual loaded spaCy model in the next cell, so it can never drift from
# the model that produced the tokens.
run_params = {
    "states": STATES,
    "pre_post": PRE_POST,
    "sample_n": SAMPLE_N,
    "sample_seed": SAMPLE_SEED,
}

In [3]:
with timed("load_corpus", log=timing_log):
    corpus_df = load_corpus(
        DATA_ROOT, states=STATES, pre_post=PRE_POST, sample_n=SAMPLE_N, seed=SAMPLE_SEED,
    )
print(f"{len(corpus_df):,} documents loaded")

with timed("load spaCy model", log=timing_log):
    nlp = spacy.load("de_core_news_lg")
run_params["preprocessing"] = f"{nlp.meta['lang']}_{nlp.meta['name']}_{nlp.meta['version']}"
print(f"preprocessing: {run_params['preprocessing']}")
preprocess = make_spacy_preprocessor(nlp)

# Cached separately from the K-scans below: this doesn't depend on K_RANGE at all, only on
# which documents were loaded and which spaCy model tokenized them.
preprocess_cache_params = {**run_params, "artifact": "tokenized_docs"}
cached_tokens = load_cache(preprocess_cache_params, suffix=".pkl")
if cached_tokens is not None:
    tokenized_docs = cached_tokens
    print(f"loaded {len(tokenized_docs):,} cached tokenized documents (skipped preprocessing)")
else:
    with timed("preprocess", log=timing_log):
        tokenized_docs = preprocess(corpus_df["text"].tolist())
    save_cache(tokenized_docs, preprocess_cache_params, suffix=".pkl")
print(f"Example tokens: {tokenized_docs[0][:10]}")

[load_corpus] 1.91s
444,189 documents loaded


[load spaCy model] 0.77s
preprocessing: de_core_news_lg_3.8.0


[preprocess] 10609.34s


Example tokens: ['herr', 'präsident', 'geehrt', 'dame', 'herr', 'mitteilen', 'ministerin', 'jutta', 'lieske', 'rücktritt']


## Vectorize and fit both models across K

Both methods share the same `tokenized_docs` and `K_RANGE`, but vectorize independently (gensim's
`Dictionary`/BoW vs. sklearn's `CountVectorizer`/doc-term matrix) since each library needs its own
input format. Results are cached under `measurement/run_history/topic_modeling/`, keyed by
`run_params` + K range + method + model seed -- rerunning this notebook with the same parameters
loads from disk instead of re-fitting.

In [4]:
with timed("build_gensim_corpus", log=timing_log):
    dictionary, gensim_corpus = build_gensim_corpus(tokenized_docs)

with timed("gensim_coherence_scan (all k)", log=timing_log):
    gensim_results = gensim_coherence_scan(
        tokenized_docs, dictionary, gensim_corpus, k_range=K_RANGE, params=run_params,
    )
gensim_results[["k", "coherence", "seconds"]]

[build_gensim_corpus] 38.18s


[gensim k=32] 352.07s, coherence=0.5013


[gensim k=34] 353.02s, coherence=0.4627


[gensim k=36] 375.83s, coherence=0.5084


[gensim k=38] 380.28s, coherence=0.4844


...

[gensim k=44] 443.52s, coherence=0.4835


[gensim k=46] 444.73s, coherence=0.4987


[gensim k=48] 478.80s, coherence=0.4872


[gensim_coherence_scan (all k)] 3679.58s


,k,coherence,seconds
0,32,0.501285,352.067889
1,34,0.462734,353.020026
2,36,0.508403,375.834098
3,38,0.484355,380.278945
4,40,0.511983,421.269932
5,42,0.464124,424.918949
6,44,0.483547,443.519124
7,46,0.498710,444.726527
8,48,0.487156,478.804670


In [5]:
with timed("build_sklearn_corpus", log=timing_log):
    vectorizer, dtm = build_sklearn_corpus(tokenized_docs)

with timed("sklearn_loglikelihood_search (all k)", log=timing_log):
    sklearn_results = sklearn_loglikelihood_search(
        dtm, vectorizer, tokenized_docs, dictionary,
        k_range=K_RANGE, params=run_params, n_iter=SKLEARN_N_ITER,
    )
sklearn_results[["k", "log_likelihood", "coherence", "seconds"]]

[build_sklearn_corpus] 12.79s


[sklearn k=46] 1043.00s, log_likelihood=-128113110.07, coherence=0.5028


[sklearn k=48] 1041.38s, log_likelihood=-128429312.35, coherence=0.4985


[sklearn k=36] 973.21s, log_likelihood=-126782106.39, coherence=0.4922


[sklearn k=40] 1003.00s, log_likelihood=-127306204.90, coherence=0.5115


...

[sklearn k=38] 992.03s, log_likelihood=-127058525.11, coherence=0.4912


[sklearn k=32] 901.79s, log_likelihood=-126090653.13, coherence=0.5022


[sklearn k=34] 956.85s, log_likelihood=-126436216.87, coherence=0.4894


[sklearn_loglikelihood_search (all k)] 8959.42s


,k,log_likelihood,coherence,seconds
0,46,-1.281131e+08,0.502848,1043.001322
1,48,-1.284293e+08,0.498474,1041.383685
2,36,-1.267821e+08,0.492248,973.210979
3,40,-1.273062e+08,0.511548,1003.004746
4,44,-1.278862e+08,0.504242,1022.540564
5,42,-1.275759e+08,0.511408,1019.559477
6,38,-1.270585e+08,0.491219,992.025967
7,32,-1.260907e+08,0.502152,901.793533
8,34,-1.264362e+08,0.489372,956.847013


## Compare the two methods' preferred K

In [6]:
best_gensim_k = int(gensim_results.loc[gensim_results["coherence"].idxmax(), "k"])
best_sklearn_k_coherence = int(sklearn_results.loc[sklearn_results["coherence"].idxmax(), "k"])
best_sklearn_k_loglik = int(sklearn_results.loc[sklearn_results["log_likelihood"].idxmax(), "k"])

print(f"gensim (c_v coherence) prefers k={best_gensim_k}")
print(f"sklearn (c_v coherence) prefers k={best_sklearn_k_coherence}")
print(f"sklearn (log-likelihood) prefers k={best_sklearn_k_loglik}")
print()
print("Same-metric comparison (coherence, both libraries):",
      "Agreement" if best_gensim_k == best_sklearn_k_coherence
      else "Disagreement -- the two implementations of the same measure land on different K")
print("Cross-metric comparison (gensim coherence vs. sklearn log-likelihood):",
      "Agreement" if best_gensim_k == best_sklearn_k_loglik
      else "Disagreement -- inspect both before picking K")

gensim (c_v coherence) prefers k=40
sklearn (c_v coherence) prefers k=40
sklearn (log-likelihood) prefers k=32

Same-metric comparison (coherence, both libraries): Agreement
Cross-metric comparison (gensim coherence vs. sklearn log-likelihood): Disagreement -- inspect both before picking K


## Inspect topics for the chosen K

In [7]:
chosen_k = best_gensim_k  # change after inspecting the comparison above

print(f"=== gensim topics (k={chosen_k}) ===\n")
chosen_gensim_model = gensim_results.set_index("k").loc[chosen_k, "model"]
for topic_id, terms in chosen_gensim_model.print_topics(num_topics=-1, num_words=10):
    print(f"Topic {topic_id}: {terms}\n")

sklearn_chosen_k = best_sklearn_k_coherence
print(f"\n=== sklearn topics (k={sklearn_chosen_k}, its own coherence-preferred K) ===\n")
chosen_sklearn_model = sklearn_results.set_index("k").loc[sklearn_chosen_k, "model"]
for topic_id, words in enumerate(sklearn_topic_words(chosen_sklearn_model, vectorizer, topn=10)):
    print(f"Topic {topic_id}: {', '.join(words)}\n")

=== gensim topics (k=40) ===



Topic 0: 0.126*"hochschule" + 0.035*"wissenschaft" + 0.032*"universität" + 0.023*"studierend" + 0.023*"forschung" + 0.023*"student" + 0.018*"wissenschaftlich" + 0.015*"studium" + 0.012*"lehre" + 0.011*"hochschulgesetz"

Topic 1: 0.017*"verfahren" + 0.013*"fall" + 0.012*"justiz" + 0.012*"enden" + 0.011*"monat" + 0.009*"richter" + 0.009*"gericht" + 0.008*"herr" + 0.008*"asylbewerber" + 0.007*"ten"

Topic 2: 0.067*"verein" + 0.062*"sport" + 0.060*"stiftung" + 0.051*"ehrenamtlich" + 0.039*"feuerwehr" + 0.028*"engagement" + 0.024*"ehrenamt" + 0.020*"verband" + 0.020*"freiwillig" + 0.014*"engagieren"

Topic 3: 0.082*"staatsregierung" + 0.051*"gebäude" + 0.039*"bau" + 0.036*"sanierung" + 0.028*"planung" + 0.026*"landesentwicklung" + 0.024*"bauen" + 0.019*"neubau" + 0.018*"brücke" + 0.017*"dringlichkeit"

Topic 4: 0.042*"sagen" + 0.041*"mal" + 0.016*"wissen" + 0.014*"glauben" + 0.013*"brauchen" + 0.012*"einfach" + 0.011*"problem" + 0.010*"reden" + 0.010*"finden" + 0.009*"ding"

Topic 5: 0.093*

Topic 0: berlin, senat, berliner, stadt, bezirk, kind, senator, thema, bürgermeister, koalition

Topic 1: herr, frage, frau, landesregierung, niedersachsen, antwort, fragen, anfrage, geehrt, präsident

Topic 2: schule, herr, digital, schüler, dame, digitalisierung, schülerin, kind, kollege, wichtig

Topic 3: unternehmen, beschäftigter, herr, mindestlohn, arbeitnehmer, öffentlich, arbeit, wirtschaft, deutschland, dame

Topic 4: flüchtling, bremen, land, deutschland, herr, mensch, asylbewerber, sicher, bremerhaven, zahl

Topic 5: gemeinde, kommunal, kommune, herr, stadt, gesetzentwurf, landkreis, frage, dame, gesetz

Topic 6: feuerwehr, wolf, ehrenamtlich, herr, freiwillig, rettungsdienst, ehrenamt, hessen, hebammen, katastrophenschutz

Topic 7: mensch, frau, leben, gesellschaft, integration, wichtig, land, behinderung, brauchen, arbeit

Topic 8: kommune, kommunal, land, landkreis, bitte, stadt, bund, gerne, spitzenverband, herr

Topic 9: stiftung, sprache, kirche, minderheit, heimat, or

## Timing summary — for estimating full-corpus runtime

This ran on the sample size and state(s) set in the Parameters cell above. Multiply the
preprocess/vectorize/fit seconds-per-document by the full pre/post-AfD corpus size (see
`preprocessing/afd_period_window.py`'s printed document count) to estimate a full run's cost
before committing to it.

In [8]:
import pandas as pd

timing_df = pd.DataFrame(timing_log)
timing_df["seconds_per_doc"] = timing_df["seconds"] / len(corpus_df)
timing_df

,step,seconds,seconds_per_doc
0,load_corpus,1.913290,0.000004
1,load spaCy model,0.766537,0.000002
2,preprocess,10609.338418,0.023885
3,build_gensim_corpus,38.178533,0.000086
4,gensim_coherence_scan (all k),3679.581645,0.008284
5,build_sklearn_corpus,12.786591,0.000029
6,sklearn_loglikelihood_search (all k),8959.422435,0.020170
